In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from NOTEBOOKS.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import teamStarPlayer, projectedStartingFive

### Load Model

In [3]:
model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MODEL.pkl')
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

### Load Player Data and Bookmaker Data

In [4]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s25= pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
df = pd.concat([s25, s26])

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

### Update projected starting lineups

In [5]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/MODELS/teamInfo.py
Updated 12 teams with confirmed lineups


### Top EVs for single bets

In [6]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 200) & (usData['ODDS'] >= -200)]

results = calculateSingleBets(df, singlePTSBookies, model, features, current_date, edge_threshold=0.20, stake=10, 
                     variance_inflation=1.1, distribution_type='t', stat_col='PTS', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

singleBets = results
singleBets = singleBets[(singleBets['SIGMA FLAG'] == 'Med') | (singleBets['SIGMA FLAG'] == 'Low')].sort_values(by='EV%', ascending=False).reset_index(drop=True)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets_{today}.csv', index=False)
singleBets.head()

Processing single bets with single model...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,RECOMMENDATION,OVER%,UNDER%,IMPLIED PROB,MODEL PROB,EDGE,EV%,KELLY_FRACTION,KELLY_DOLLARS,CONFIDENCE INTERVAL,INTERVAL WIDTH,SIGMA,SIGMA FLAG,EXPECTED ROI,SIMULATION_METHOD
0,Goga Bitadze,BetMGM,player_points,4.5,105,Over,11.21,1,0.911,0.089,0.488,0.911,0.423,8.67,0.826,2.62,"(0.3, 22.1)",21.73,5.54,Med,0.87,Monte Carlo
1,Marvin Bagley III,BetMGM,player_points,5.5,-140,Over,8.80,0,0.742,0.258,0.583,0.742,0.159,2.73,0.382,1.79,"(0.0, 20.5)",20.49,5.96,Med,0.27,Monte Carlo
2,Tari Eason,BetMGM,player_points,10.5,-118,Over,12.74,0,0.673,0.327,0.541,0.673,0.132,2.44,0.288,2.12,"(1.2, 24.2)",23.00,5.87,Med,0.24,Monte Carlo
3,Tari Eason,BetOnline.ag,player_points,10.5,-123,Over,12.74,0,0.673,0.327,0.552,0.673,0.122,2.21,0.271,2.03,"(1.2, 24.2)",23.00,5.87,Med,0.22,Monte Carlo
4,Tari Eason,Bovada,player_points,11.5,105,Over,12.74,0,0.594,0.406,0.488,0.594,0.106,2.18,0.208,2.18,"(1.2, 24.2)",23.00,5.87,Med,0.22,Monte Carlo


## Top EVs for 2 leg bets

### Underdog picks

In [15]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=100, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results
# underdogPairs = underdogPairs[
#     underdogPairs[['sigma_flag1', 'sigma_flag2']].isin(['Med', 'Low']).all(axis=1)
# ].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs_{today}.csv', index=False)
underdogPairs.head()

,player1,player2,line1,line2,pred1,pred2,model_side1,model_side2,prob1,prob2,prob_both,edge1,edge2,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,interval_width1,interval_width2,sigma1,sigma2,sigma_flag1,sigma_flag2,simulation_method
0,Giannis Antetokounmpo,Domantas Sabonis,32.5,16.5,34.06,17.43,over,over,0.596,0.551,0.2954,0.018,-0.027,-0.005,-0.11,0.000,0,"(19.8, 48.3)","(2.2, 32.6)",28.53,30.42,7.28,7.76,High,High,Monte Carlo
1,Giannis Antetokounmpo,DeMar DeRozan,32.5,20.5,34.06,20.93,over,over,0.596,0.521,0.2795,0.018,-0.057,-0.020,-0.16,0.000,0,"(19.8, 48.3)","(6.0, 35.9)",28.53,29.87,7.28,7.62,High,High,Monte Carlo
2,Giannis Antetokounmpo,Russell Westbrook,32.5,12.5,34.06,14.59,over,over,0.596,0.637,0.3416,0.018,0.059,0.038,0.02,0.012,0,"(19.8, 48.3)","(0.9, 28.2)",28.53,27.29,7.28,6.96,High,High,Monte Carlo
3,Giannis Antetokounmpo,LaMelo Ball,32.5,25.5,34.06,28.11,over,over,0.596,0.637,0.3415,0.018,0.059,0.038,0.02,0.012,0,"(19.8, 48.3)","(11.0, 45.2)",28.53,34.20,7.28,8.72,High,High,Monte Carlo
4,Giannis Antetokounmpo,Julius Randle,32.5,24.5,34.06,23.72,over,under,0.596,0.560,0.3000,0.018,-0.018,-0.000,-0.10,0.000,0,"(19.8, 48.3)","(9.7, 37.8)",28.53,28.11,7.28,7.17,High,High,Monte Carlo


### Prizepicks picks

In [16]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=100, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)
pairsPrizepicks = results
# pairsPrizepicks = pairsPrizepicks[
#     pairsPrizepicks[['sigma_flag1', 'sigma_flag2']].isin(['Med', 'Low']).all(axis=1)
# ].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs_{today}.csv', index=False)
pairsPrizepicks.head()

,player1,player2,line1,line2,pred1,pred2,model_side1,model_side2,prob1,prob2,prob_both,edge1,edge2,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,interval_width1,interval_width2,sigma1,sigma2,sigma_flag1,sigma_flag2,simulation_method
0,Giannis Antetokounmpo,Zach LaVine,31.5,23.5,34.06,26.67,over,over,0.661,0.673,0.4001,0.083,0.095,0.089,0.20,0.100,0,"(19.8, 48.3)","(10.3, 43.0)",28.53,32.68,7.28,8.34,High,High,Monte Carlo
1,Giannis Antetokounmpo,DeMar DeRozan,31.5,19.5,34.06,20.93,over,over,0.661,0.582,0.3461,0.083,0.004,0.043,0.04,0.019,0,"(19.8, 48.3)","(6.0, 35.9)",28.53,29.87,7.28,7.62,High,High,Monte Carlo
2,Giannis Antetokounmpo,Domantas Sabonis,31.5,16.5,34.06,17.43,over,over,0.661,0.551,0.3277,0.083,-0.027,0.028,-0.02,0.000,0,"(19.8, 48.3)","(2.2, 32.6)",28.53,30.42,7.28,7.76,High,High,Monte Carlo
3,Giannis Antetokounmpo,Russell Westbrook,31.5,12.5,34.06,14.59,over,over,0.661,0.637,0.3790,0.083,0.059,0.071,0.14,0.069,0,"(19.8, 48.3)","(0.9, 28.2)",28.53,27.29,7.28,6.96,High,High,Monte Carlo
4,Giannis Antetokounmpo,Keon Ellis,31.5,7.5,34.06,9.85,over,over,0.661,0.650,0.3865,0.083,0.072,0.077,0.16,0.080,0,"(19.8, 48.3)","(0.0, 23.9)",28.53,23.93,7.28,7.18,High,High,Monte Carlo


## 3 leg parlay

### Underdog picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.10, stake=100, 
                     variance_inflation=1.1, distribution_type='t', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg[
    threeLeg[['sigma_flag1', 'sigma_flag2', 'sigma_flag3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios_{today}.csv', index=False)
underdogTrios.head()

,player1,player2,player3,line1,line2,line3,pred1,pred2,pred3,model_side1,model_side2,model_side3,prob1,prob2,prob3,prob_all_three,edge1,edge2,edge3,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,confidence_interval3,interval_width1,interval_width2,interval_width3,sigma1,sigma2,sigma3,sigma_flag1,sigma_flag2,sigma_flag3,simulation_method


### Prizepicks picks

In [10]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.40, stake=100, 
                     variance_inflation=1.1, distribution_type='normal', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg[
    threeLeg[['sigma_flag1', 'sigma_flag2', 'sigma_flag3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios_{today}.csv', index=False)
triosPrizepicks.head()

,player1,player2,player3,line1,line2,line3,pred1,pred2,pred3,model_side1,model_side2,model_side3,prob1,prob2,prob3,prob_all_three,edge1,edge2,edge3,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,confidence_interval3,interval_width1,interval_width2,interval_width3,sigma1,sigma2,sigma3,sigma_flag1,sigma_flag2,sigma_flag3,simulation_method


In [11]:
def count_line_hits(player_df, line, category, game_windows=[5, 10, 15]):
    results = {}
    player_df_sorted = player_df.sort_values('GAME_DATE')
    total_games = len(player_df_sorted)

    for window in game_windows:
        # Handle players with fewer games
        if total_games < window:
            last_n_games = player_df_sorted
        else:
            last_n_games = player_df_sorted.tail(window)

        if category == 'player_points':
            hits = (last_n_games['PTS'] > line).sum()
        elif category == 'player_assists':
            hits = (last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds':
            hits = (last_n_games['REB'] > line).sum()
        elif category == 'player_threes':
            hits = (last_n_games['FG3M'] > line).sum()
        elif category == 'player_blocks':
            hits = (last_n_games['BLK'] > line).sum()
        elif category == 'player_steals':
            hits = (last_n_games['STL'] > line).sum()
        elif category == 'player_field_goals':
            hits = (last_n_games['FGM'] > line).sum()
        elif category == 'player_frees_made':
            hits = (last_n_games['FTM'] > line).sum()
        elif category == 'player_points_rebounds_assists':
            hits = (last_n_games['PTS'] + last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_points_rebounds':
            hits = (last_n_games['PTS'] + last_n_games['REB'] > line).sum()
        elif category == 'player_points_assists':
            hits = (last_n_games['PTS'] + last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds_assists':
            hits = (last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_turnovers':
            hits = (last_n_games['TOV'] > line).sum()
        else:
            hits = 0

        results['NAME'] = player_df_sorted['PLAYER_NAME'].iloc[0] if total_games > 0 else 'Unknown'
        results['CATEGORY'] = category
        results['LINE'] = line
        results[f'LAST {window}'] = hits
        results[f'HIT RATE % LAST {window}'] = round(hits / window, 2)


    return results

prizePicks = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks')]
prizePicks= prizePicks.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
prizePicks

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
150,PrizePicks,player_points,Giannis Antetokounmpo,Over,31.5,-137,2025-11-01,2025-11-01T15:04:03Z
152,PrizePicks,player_points,Zach LaVine,Over,23.5,-137,2025-11-01,2025-11-01T15:04:03Z
154,PrizePicks,player_points,DeMar DeRozan,Over,19.5,-137,2025-11-01,2025-11-01T15:04:03Z
156,PrizePicks,player_points,Domantas Sabonis,Over,16.5,-137,2025-11-01,2025-11-01T15:04:03Z
158,PrizePicks,player_points,Ryan Rollins,Over,14.5,-137,2025-11-01,2025-11-01T15:04:03Z
...,...,...,...,...,...,...,...,...
2247,PrizePicks,player_blocks_steals,Alperen Sengun,Over,1.5,-137,2025-11-02,2025-11-01T15:04:38Z
2249,PrizePicks,player_blocks_steals,Josh Minott,Over,1.5,-137,2025-11-02,2025-11-01T15:04:38Z
2251,PrizePicks,player_blocks_steals,Kevin Durant,Over,1.5,-137,2025-11-02,2025-11-01T15:04:38Z
2253,PrizePicks,player_blocks_steals,Josh Okogie,Over,1.5,-137,2025-11-02,2025-11-01T15:04:38Z


In [12]:
line_hit_data = []

for index, row in prizePicks.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

        
        

Saved 74 records for player_points to player_points.csv
Saved 59 records for player_rebounds to player_rebounds.csv
Saved 31 records for player_assists to player_assists.csv
Saved 10 records for player_threes to player_threes.csv
Saved 8 records for player_blocks to player_blocks.csv
Saved 15 records for player_steals to player_steals.csv
Saved 22 records for player_field_goals to player_field_goals.csv
Saved 27 records for player_frees_made to player_frees_made.csv
Saved 76 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 75 records for player_points_rebounds to player_points_rebounds.csv
Saved 68 records for player_points_assists to player_points_assists.csv
Saved 41 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 17 records for player_turnovers to player_turnovers.csv
Saved 12 records for player_blocks_steals to player_blocks_steals.csv

All category files saved to ../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS


In [13]:
underdog = dfsData[(dfsData['BOOKMAKER'] == 'Underdog')]
underdog= underdog.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
line_hit_data = []

for index, row in underdog.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

Saved 68 records for player_points to player_points.csv
Saved 26 records for player_rebounds to player_rebounds.csv
Saved 11 records for player_assists to player_assists.csv
Saved 9 records for player_threes to player_threes.csv
Saved 1 records for player_blocks to player_blocks.csv
Saved 75 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 33 records for player_points_rebounds to player_points_rebounds.csv
Saved 26 records for player_points_assists to player_points_assists.csv
Saved 18 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 7 records for player_turnovers to player_turnovers.csv
Saved 1 records for player_blocks_steals to player_blocks_steals.csv

All category files saved to ../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG
